In [ ]:
import json
import os

import cv2
import matplotlib.pyplot as plt
import matplotlib.lines as mlines

from src.gait_analysis.data_cleaning.data_smoothing import smooth_pose_data
from src.gait_analysis.parameter_calculation.visualize import visualize_gait_phase
from src.gait_analysis.parameter_calculation.kinematic_parameters.angles import (
    calculate_kinematic_angles,
)
from src.gait_analysis.parameter_calculation.biomecanical_parameters.ground_contact_time import (
    calculate_gct,
)
from src.gait_analysis.parameter_calculation.biomecanical_parameters.ground_contact_time import (
    calculate_flight_times,
)
from src.gait_analysis.parameter_calculation.kinematic_parameters.step_parameters import (
    calculate_step_metrics,
)

In [ ]:
model_id = '000332'
video_key = 'v1'

overlay_path = (
    f"../data/output/tuned_yolo_overlays/{model_id}/outdoor/RES1080_FPS60_LENlinear/{video_key}_overlay.mp4"
)
pose_model_output_path = (
    f"../data/output/tuned_yolo/{model_id}/outdoor/RES1080_FPS60_LENlinear/{video_key}.json"
)

In [ ]:
cap = cv2.VideoCapture(overlay_path)
FPS = cap.get(cv2.CAP_PROP_FPS)

In [ ]:
with open(os.path.join(pose_model_output_path), "r") as f:
    pose_model_output = json.load(f)

connections = pose_model_output["connections"]
pose_data = pose_model_output["pose_data"]

In [ ]:
keypoints = list(set([x for xs in connections for x in xs]))
keys_to_exclude = {"raw_keypoints"}

smooth_data = smooth_pose_data(
    pose_data=pose_data,
    keypoints=keypoints,
    keys_to_exclude=keys_to_exclude,
    fps=FPS,
    cutoff=4.0,
)

# 1. KINEMATIC PARAMETERS - ANGLES

* **Joint Angles (Hip, Knee, Ankle)**
  These angles are derived by first calculating the interior angle at the joint vertex. This is done using the dot product of the two vectors formed by the three connected skeletal keypoints:

  $$\cos(\theta) = \frac{\mathbf{v_1} \cdot \mathbf{v_2}}{|\mathbf{v_1}| |\mathbf{v_2}|}$$

  Once the baseline interior angle is computed, it is transformed to represent standard biomechanical extensions/flexions:
  * **Hip angle:** 180° minus the interior angle formed by the shoulder, hip (vertex), and knee.
  * **Knee angle:** 180° minus the interior angle formed by the hip, knee (vertex), and ankle.
  * **Ankle angle:** 90° minus the interior angle formed by the knee, ankle (vertex), and big toe.

* **Trunk lean angle**
  This metric evaluates the forward or backward tilt of the torso relative to a true vertical line. It is calculated using the arctangent function (`arctan2`) on the horizontal and vertical distances between the shoulder and the hip keypoints. To ensure consistency across different videos, the final angle is directionally adjusted based on the subject's overall movement vector. This direction is defined by comparing the hip's horizontal position (x-coordinate) in the final frame of the sequence to its position in the first frame. If the final x-coordinate is not greater than the initial—indicating the subject is moving right-to-left—the sign of the calculated angle is inverted.

In [ ]:
right_angles = calculate_kinematic_angles(smooth_data, side="right")
left_angles = calculate_kinematic_angles(smooth_data, side="left")

In [ ]:
fig, axes = plt.subplots(nrows=4, ncols=1, figsize=(14, 12), sharex=True)

right_angles.plot(
    x="timestamp_ms", y="right_hip_angle", ax=axes[0], color="tab:blue", grid=True
)
axes[0].set_ylabel("Hip Flexion (°)")

right_angles.plot(
    x="timestamp_ms", y="right_knee_angle", ax=axes[1], color="tab:orange", grid=True
)
axes[1].set_ylabel("Knee Flexion (°)")

right_angles.plot(
    x="timestamp_ms", y="right_ankle_angle", ax=axes[2], color="tab:green", grid=True
)
axes[2].set_ylabel("Ankle Flexion (°)")

right_angles.plot(
    x="timestamp_ms", y="right_trunk_lean", ax=axes[3], color="tab:red", grid=True
)
axes[3].set_ylabel("Trunk Lean (°)")
axes[3].set_xlabel("Time (ms)")

plt.suptitle("Angles, (°), vs Timestamp, [ms] - RIGHT side")
plt.tight_layout()
plt.show()

# 2. BIOMECHANICAL PARAMETERS

* **Ground contact time (GCT)**
  This metric calculates the duration a foot remains in contact with the ground during a single stride. The algorithm determines this by analyzing the horizontal (x-axis) positions of the heel and big toe relative to the hip. First, it accounts for the subject's overall running direction to ensure accurate relative positioning. It then identifies the frame of foot-strike (landing) using a peak-detection algorithm to find the maximum forward extension of the heel relative to the hip. To determine the liftoff frame, the algorithm searches forward from the landing point to locate the first instance where the big toe passes behind the hip (meaning its relative horizontal position becomes negative). The time elapsed between these two frames represents the GCT.

* **Flight time**
  This metric measures the aerial phase of the running gait cycle, representing the duration where neither foot is touching the ground. It is calculated by aggregating all computed landing and liftoff events from both the left and right feet into a single, chronologically sorted timeline. The algorithm iterates through this unified timeline and identifies an aerial phase whenever a liftoff event from one foot is immediately followed by a landing event from the contralateral (opposite) foot. The time difference between these two consecutive events constitutes the flight time.

In [ ]:
left_gct = calculate_gct(smooth_data, fps=FPS, side="left")
right_gct = calculate_gct(smooth_data, fps=FPS, side="right")

In [ ]:
side = "left"

if side == "left":
    gct_records = left_gct
else:
    gct_records = right_gct

smooth_data[f"{side}_heel_rel_x"] = (
    smooth_data[f"{side}_heel_x"] - smooth_data[f"{side}_hip_x"]
)
smooth_data[f"{side}_toe_rel_x"] = (
    smooth_data[f"{side}_big_toe_x"] - smooth_data[f"{side}_hip_x"]
)

direction = (
    1
    if smooth_data[f"{side}_hip_x"].iloc[-1] > smooth_data[f"{side}_hip_x"].iloc[0]
    else -1
)

smooth_data[f"{side}_heel_rel_x"] *= direction
smooth_data[f"{side}_toe_rel_x"] *= direction

landings = gct_records["landing_time"]
liftoffs = gct_records["liftoff_time"]

landing_points = smooth_data[smooth_data["timestamp_ms"].isin(landings)]
liftoff_points = smooth_data[smooth_data["timestamp_ms"].isin(liftoffs)]

fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(12, 10), sharex=True)


ax1.plot(
    smooth_data["timestamp_ms"],
    smooth_data[f"{side}_heel_rel_x"],
    label="Heel Relative X (Heel - Hip)",
    color="tab:blue",
    linewidth=2,
)

ax1.scatter(
    landing_points["timestamp_ms"],
    landing_points[f"{side}_heel_rel_x"],
    color="red",
    marker="x",
    s=100,
    linewidths=3,
    label="Landing (Max Forward)",
    zorder=5,
)

ax1.axhline(0, color="black", linestyle="--", alpha=0.3, label="Hip Center")

ax1.set_ylabel(
    "Relative Position [px]\n(Positive = In Front of Body)", color="tab:blue"
)
ax1.set_title(f"{side.capitalize()} Heel: Relative Horizontal Position")
ax1.legend(loc="upper right")
ax1.grid(True, alpha=0.3)

ax2.plot(
    smooth_data["timestamp_ms"],
    smooth_data[f"{side}_toe_rel_x"],
    label="Toe Relative X (Toe - Hip)",
    color="tab:green",
    linewidth=2,
)

ax2.scatter(
    liftoff_points["timestamp_ms"],
    liftoff_points[f"{side}_toe_rel_x"],
    color="magenta",
    marker="o",
    s=100,
    linewidths=2,
    label="Liftoff (Max Backward)",
    zorder=5,
)

ax2.axhline(0, color="black", linestyle="--", alpha=0.3, label="Hip Center")

ax2.set_xlabel("Time (ms)")
ax2.set_ylabel("Relative Position [px]\n(Negative = Behind Body)", color="tab:green")
ax2.set_title(f"{side.capitalize()} Big Toe: Relative Horizontal Position")
ax2.legend(loc="upper right")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Left side is marked blue, right -- red.

In [ ]:
landing_idx = 0
landing_time = gct_records["landing_time"][landing_idx]
target_frame_idx = int(round((landing_time / 1000.0) * FPS))

visualize_gait_phase(
    overlay_path,
    target_frame_idx,
    landing_time,
    1,
    figsize=(25, 10),
    gait_phase=f"LANDING - {side.upper()}",
)

In [ ]:
landing_idx = 3
landing_time = gct_records["liftoff_time"][landing_idx]
target_frame_idx = int(round((landing_time / 1000.0) * FPS))

visualize_gait_phase(
    overlay_path,
    target_frame_idx,
    landing_time,
    1,
    figsize=(25, 10),
    gait_phase=f"LIFTOFF - {side.upper()}",
)

In [ ]:
right_gct

In [ ]:
left_gct

In [ ]:
calculate_flight_times(left_gct, right_gct)

In [ ]:
fig, axes = plt.subplots(nrows=4, ncols=1, figsize=(14, 12), sharex=True)

right_angles.plot(
    x="timestamp_ms", y="right_hip_angle", ax=axes[0], color="tab:blue", grid=True
)
axes[0].set_ylabel("Hip Flexion (°)")

right_angles.plot(
    x="timestamp_ms", y="right_knee_angle", ax=axes[1], color="tab:orange", grid=True
)
axes[1].set_ylabel("Knee Flexion (°)")

right_angles.plot(
    x="timestamp_ms", y="right_ankle_angle", ax=axes[2], color="tab:green", grid=True
)
axes[2].set_ylabel("Ankle Flexion (°)")

right_angles.plot(
    x="timestamp_ms", y="right_trunk_lean", ax=axes[3], color="tab:red", grid=True
)
axes[3].set_ylabel("Trunk Lean (°)")
axes[3].set_xlabel("Time (ms)")


for ax in axes:
    for _, row in right_gct.iterrows():
        land_t = row["landing_time"]
        lift_t = row["liftoff_time"]

        ax.axvline(x=land_t, color="red", linestyle="--", alpha=0.8, linewidth=1.5)
        ax.axvline(x=lift_t, color="magenta", linestyle="--", alpha=0.8, linewidth=1.5)

        ax.axvspan(land_t, lift_t, color="gray", alpha=0.15)

landing_line = mlines.Line2D([], [], color="red", linestyle="--", label="Landing (IC)")
liftoff_line = mlines.Line2D(
    [], [], color="magenta", linestyle="--", label="Liftoff (TO)"
)
stance_patch = plt.Rectangle(
    (0, 0), 1, 1, fc="gray", alpha=0.15, label="Ground Contact"
)

axes[0].legend(
    handles=[
        axes[0].get_legend_handles_labels()[0][0],
        landing_line,
        liftoff_line,
        stance_patch,
    ],
    loc="upper right",
)
axes[1].get_legend().remove()
axes[2].get_legend().remove()
axes[3].get_legend().remove()

plt.suptitle("Angles (°), vs Timestamp [ms] - RIGHT side", y=0.98, fontsize=14)
plt.tight_layout()
plt.show()


| RLA Phase | Hip Kinematics & Muscles | Knee Kinematics & Muscles | Ankle Kinematics & Muscles |
| :--- | :--- | :--- | :--- |
| **Initial Contact** | ~20° Flexion <br> *(Gluteus Maximus, Hamstrings)* | ~0-5° Flexion <br> *(Quadriceps)* | ~0° (Neutral) <br> *(Tibialis Anterior)* |
| **Loading Response** | ~20° Flexion <br> *(Gluteus Maximus, Hamstrings)* | ~15° Flexion <br> *(Quadriceps eccentric)* | ~5-10° Plantarflexion <br> *(Tibialis Anterior eccentric)* |
| **Midstance** | Moving to 0° (Neutral) <br> *(Iliopsoas eccentric)* | ~0-5° Flexion <br> *(Quadriceps)* | ~5° Dorsiflexion <br> *(Calf muscles eccentric)* |
| **Terminal Stance** | ~10-20° Extension <br> *(None/Passive)* | ~0-5° Flexion <br> *(None/Passive)* | ~10° Dorsiflexion <br> *(Calf muscles concentric)* |
| **Pre-Swing** | ~10° Extension <br> *(Adductor Longus)* | ~40° Flexion <br> *(Popliteus, passive flexion)* | ~15-20° Plantarflexion <br> *(Calf muscles)* |
| **Initial Swing** | ~15° Flexion <br> *(Iliopsoas)* | ~60° Flexion <br> *(Hamstrings)* | ~5-10° Plantarflexion <br> *(Tibialis Anterior)* |
| **Midswing** | ~25° Flexion <br> *(Hamstrings eccentric)* | ~25° Flexion <br> *(Hamstrings eccentric)* | ~0° (Neutral) <br> *(Tibialis Anterior)* |
| **Terminal Swing** | ~20° Flexion <br> *(Hamstrings eccentric)* | ~0-5° Flexion <br> *(Quadriceps)* | ~0° (Neutral) <br> *(Tibialis Anterior)* ||


source: https://www.physio-pedia.com/The_Gait_Cycle

# 3. KINEMATIC PARAMETERS - STEP METRICS

* **Step frequency**
  (cadence) evaluates the rate of stepping in steps per minute (SPM). It is determined by first constructing a unified, chronological timeline of all foot landing events. The step time is calculated as the duration (in milliseconds) between these two consecutive landings. Finally, step frequency is derived by dividing 60,000 by this step time.

* **Step length**
  This metric measures the spatial distance of a single step at the precise moment of foot-strike. It is calculated using two distinct methods: measuring the absolute horizontal distance (in pixels) between the left and right heels, and separately between the left and right big toes. To translate these pixel measurements into real-world meters, a dynamic scaling ratio is applied. This ratio is established by dividing the runner's known physical height by the vertical pixel height of the runner's bounding box (`bbox_h`) in that specific frame. Both the heel-to-heel and toe-to-toe metrics are recorded.

* **Running speed**
  This evaluates the subject's velocity for each individual step. It is calculated directly from the two preceding metrics by dividing the step length (heel version, in meters) by the step time (converted to seconds), resulting in a base speed of meters per second (m/s). This base speed is also converted into kilometers per hour (km/h) and a standard running pace (minutes per kilometer).

In [ ]:
height = 1.8

In [ ]:
step_metrics = calculate_step_metrics(
    left_gct, right_gct, smooth_data, runner_height_m=height
)

step_metrics

In [ ]:
step_metrics.drop(columns=["landing_time", "landing_foot"]).mean()